<a href="https://colab.research.google.com/github/Nessynw/election-polarization-analysis/blob/main/fondement.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import numpy as np

"""
    Génère un profil dans A^n avec contrôle de polarisation.
    polarization=0 -> profil pa (tous mêmes bulletins)
    polarization=1 -> profil pa,a (moitié a, moitié a_op)
    0<p<1 -> chaque votante choisit a_op avec probabilité p
    noise -> probabilité de retourner individuellement chaque bit (optionnel)
    """
def generate_approval_profile(n, m, polarization=0.0, noise=0.0, r=None):
    if n % 2 != 0 or m % 2 != 0:
        raise ValueError("n et m doivent être pairs")
    if r is None:
        r = np.random.default_rng()
    a = r.integers(0, 2, size=m)
    a_op = 1 - a
    profile = np.empty((n, m), dtype=int)
    for i in range(n):
        base = a_op if (polarization == 1.0 and i >= n//2) or (0 < polarization < 1 and r.random() < polarization) else a
        if noise > 0:
            flips = r.random(m) < noise
            profile[i] = np.where(flips, 1 - base, base)
        else:
            profile[i] = base.copy()
    return profile

# exemple rapide
if __name__ == "__main__":
    # Changed m from 5 to 4 to satisfy the even number requirement
    prof = generate_approval_profile(n=10, m=4, polarization=0.7, noise=0.05, r=np.random.default_rng(42))
    print(prof)


[[0 1 1 0]
 [0 1 1 0]
 [1 0 0 1]
 [1 0 0 1]
 [0 1 1 0]
 [1 0 0 1]
 [1 0 0 1]
 [1 0 0 1]
 [0 1 1 0]
 [1 0 0 0]]


In [ ]:
import numpy as np
"""
    Génère un profil dans L^n avec contrôle de polarisation.
    - polarization=0   -> profil p⪰ (toutes les votantes ont le même ordre)
    - polarization=1   -> profil p⪰,⪰ (moitié ordre de base, moitié ordre opposé)
    - 0 < p < 1        -> chaque votante prend l'ordre opposé avec probabilité p
    - noise            -> petits swaps adjacents pour casser les clones exacts (optionnel)
    """
def generate_rank_profile(n, m, polarization=0.0, noise=0.0, r=None):
    if n % 2 != 0 or m % 2 != 0:
        raise ValueError("n et m doivent être pairs")
    if r is None:
        r = np.random.default_rng()

    base = r.permutation(np.arange(1, m + 1))  # ordre de référence (1..m)
    opposite = m - base + 1                    # ordre opposé exact

    profile = np.empty((n, m), dtype=int)
    for i in range(n):
        order = opposite if (polarization == 1.0 and i >= n // 2) or (0 < polarization < 1 and r.random() < polarization) else base

        if noise > 0:
            # appliquer k petits swaps adjacents
            k = max(1, int(round(noise * m)))
            tmp = order.copy()
            for _ in range(k):
                j = r.integers(0, m - 1)
                tmp[j], tmp[j + 1] = tmp[j + 1], tmp[j]
            profile[i] = tmp
        else:
            profile[i] = order.copy()

    return profile

# exemple rapide
if __name__ == "__main__":
    # Changed m from 5 to 4 to satisfy the even number requirement
    prof = generate_rank_profile(n=10, m=4, polarization=0.7, noise=0.05, r=np.random.default_rng(42))
    print(prof)


[[4 2 3 1]
 [2 1 3 4]
 [3 4 2 1]
 [4 3 1 2]
 [1 2 4 3]
 [1 3 2 4]
 [4 2 3 1]
 [1 3 2 4]
 [1 2 4 3]
 [2 1 3 4]]


In [2]:
import itertools
import numpy as np

def paire_approbation(profile: np.ndarray):
    """
    profile : array (n, m) de 0/1 (A^n)
    Retourne un dict {(i,j): d_ckcl(p)} pour toutes les paires i<j.
    d_ij = |#(i approuvé, j non) - #(j approuvé, i non)|
    """
    n, m = profile.shape
    diffs = {}
    for i, j in itertools.combinations(range(m), 2):
        pref_i = np.sum((profile[:, i] == 1) & (profile[:, j] == 0))
        pref_j = np.sum((profile[:, j] == 1) & (profile[:, i] == 0))
        diffs[(i, j)] = abs(int(pref_i - pref_j))
    return diffs

def paire_ranking(profile: np.ndarray):
    """
    profile : array (n, m) de rangs (1..m ou 0..m-1) (L^n)
    Retourne un dict {(i,j): d_ckcl(p)} pour toutes les paires i<j.
    d_ij = |#(i classé avant j) - #(j classé avant i)|
    """
    n, m = profile.shape
    diffs = {}
    for i, j in itertools.combinations(range(m), 2):
        pref_i = np.sum(profile[:, i] < profile[:, j])
        pref_j = np.sum(profile[:, j] < profile[:, i])
        diffs[(i, j)] = abs(int(pref_i - pref_j))
    return diffs

if __name__ == "__main__":
    # approvals
    pA = np.array([[1,0,1],[0,1,0],[1,0,0],[0,1,1]])
    print(paire_approbation(pA))
    # rankings (rangs 1..m)
    pL = np.array([[1,2,3],[3,1,2],[2,3,1],[2,1,3]])
    print(paire_ranking(pL))


{(0, 1): 0, (0, 2): 0, (1, 2): 0}
{(0, 1): 0, (0, 2): 0, (1, 2): 2}


In [5]:
#question8
def hamming(a,b): #a et b sont des bulletins de taille m (vote par approbations)
    if (len(a)!=len(b)):
        print("Un ou plusieurs bulletins n'ont pas la bonne taille")
        return
    d=0
    for i in range(len(a)):
        if (a[i]!=b[i]):
            d+=1
    return d

def spearman(a,b): # (vote par ordres totaux)
    if (len(a)!=len(b)):
        print("Un ou plusieurs bulletins n'ont pas la bonne taille")
        return
    d=0
    for i in range(len(a)):
        d+=abs(a[i]-b[i])
    return d

In [4]:
#question12
import numpy as np
from scipy.optimize import linear_sum_assignment

def u1ParApprobation(profile):
    n=len(profile)
    m=len(profile[0])
    sol=np.zeros(m) #bulletin solution de u1* de taille m
    nb1=np.zeros(m) #somme de chaque bulletin
    for j in range(m):
        for i in range(n):
            nb1[j]+=profile[i,j]
        if (nb1[j]<=n/2): #si pour le candidat j il y a plus de votes 0 que de votes 1
            sol[j]=0
        else:
            sol[j]=1
    return sol

def u1ParOrdresTotaux(profile):
    n=len(profile)
    m=len(profile[0])
    sol=np.zeros(m) #bulletin solution de u1* de taille m
    couts=np.zeros((m,m)) #matrice des poids
    for j in range(m):
        for pos in range(m):
            couts[j,pos]=np.sum(np.abs(pos+1-profile[:,j]))
    row_ind, col_ind = linear_sum_assignment(couts) #pour chaque indice de ligne rowind [0,1,2,..m] on associe l'indice de colonne optimal colind
    sol[row_ind]=col_ind+1
    return sol

profilApprobations=np.array([[1,0,0,1],[0,1,0,1],[1,0,0,1],[0,1,1,1]])
print(u1ParApprobation(profilApprobations))

profilOrdresTotaux=np.array([[1,3,2,4],[1,2,3,4],[4,1,2,3],[1,2,3,4]])
print(u1ParOrdresTotaux(profilOrdresTotaux))

[0. 0. 0. 1.]
[1. 2. 3. 4.]


In [ ]:
#question 14
def phiParApprobation(profil):
    n=len(profil)
    m=len(profil[0])
    phi=2/(n*m)*(u1ParApprobation(profil)-u2ParApprobation(profil))
    return phi

def phiParOrdresTotaux(profil):
    n=len(profil)
    m=len(profil[0])
    phi=4/(n*m*m)*(u1ParApprobation(profil)-u2ParApprobation(profil))
    return phi

In [ ]:
#question 15
import matplotlib.pyplot as plt

def phiEvolution(n,m):
    polar=[0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1]
    phiA=[]
    phiL=[]
    for i in range(len(polar)):
        profilA=generate_approval_profile(n,m,polar[i])
        profilL=generate_rank_profile(n,m,polar[i])
        phiA.append(phiParApprobation(profilA))
        phiL.append(phiParOrdresTotaux(profilL))
    plt.plot(polar,phiA)
    plt.plot(polar,phiL)